# Challenge 4: Trajectory evals

A **trajectory eval** scores the *sequence of tool calls* instead of the answer. It catches agents that reach a plausible answer by a wrong or wasteful path, for example by skipping a required check.

Four evaluators:

| Evaluator | Question |
|---|---|
| `exact_match` | Same tools, same order? |
| `extra_steps` | How many tool calls beyond the expected ones? |
| `missing_steps` | How many expected calls were skipped? |
| `in_order` | Do the expected calls appear in order, ignoring extras? |

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import Client, uuid7

import lab_eval
from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS, build_agent, extract_tool_calls

client = Client()
DATASET = lab_eval.dataset_name()
print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", DATASET)

## Extract the trajectory

`extract_tool_calls` walks the message list and collects each tool name in order. The target returns it as `trajectory`. Run it on one example first.

In [ ]:
agent = build_agent()


def run_agent(inputs: dict) -> dict:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config={"configurable": {"thread_id": str(uuid7())}},
    )
    return {"response": result["messages"][-1].text,
            "trajectory": extract_tool_calls(result["messages"])}


out = run_agent({"query": "Refund order A1002."})
print("actual  :", out["trajectory"])
print("expected: ['lookup_order', 'check_refund_eligibility']")

**Result:** the actual trajectory for a refund the agent should decline. Compare it to the expected one.

## The evaluators

`exact_match` and `missing_steps` are provided. `missing_steps` uses `collections.Counter` to diff the two lists as multisets:

In [ ]:
from collections import Counter


def exact_match(outputs, reference_outputs):
    return {"key": "exact_match", "score": int(outputs["trajectory"] == reference_outputs["trajectory"])}


def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


def in_order(outputs, reference_outputs):
    it = iter(outputs["trajectory"])
    return {"key": "in_order", "score": int(all(step in it for step in reference_outputs["trajectory"]))}

`extra_steps` is the mirror image of `missing_steps`: subtract the other way. The next cell holds this code:

```python
def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

## Run it

This is the **baseline** run for the eval-driven loop in challenge 5. It takes about a minute.

In [ ]:
results = client.evaluate(
    run_agent,
    data=DATASET,
    evaluators=[exact_match, extra_steps, missing_steps, in_order],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
lab_eval.compare(prefixes=("trajectory",))

## Find the failures

Open the experiment in LangSmith and sort by `exact_match`. For each row scoring 0, compare the actual and expected `trajectory`. Look at the refund requests, and ask whether `issue_refund` ran without `check_refund_eligibility` first.

**Result:** `exact_match` below 1.00 for the baseline agent, with `missing_steps` above 0 on the refund rows. The agent's answers looked fine, but its process broke a rule, and only the trajectory eval shows it.